In [45]:
import pandas as pd
import numpy as np
import optimization as opt

Original data has 23232000 rows.

In [14]:
iland_cluster6_chunks = pd.read_csv('cluster6_for_Konstantin.csv.gz', chunksize=10000)
iland_cluster6 = pd.concat([chunk[(chunk['RCPScenario'] != 'historical') & (chunk['ClimateModel'] == 'ICHEC-EC-EARTH') & (chunk['Replicate'] == 'repl0')] for chunk in iland_cluster6_chunks])

In [10]:
iland_cluster6.head()

,decade,rid,Cluster,ClimateModel,RCPScenario,Management,Replicate,abovegroundCarbon,volume,crownCover,...,Rh,NEP,cumNPP,cumRh,cumNEP,evapotranspiration_mm,mean_soilwatercontent_mm,mean_soilwatercontent_growingseason_mm,volumeHarvested,volumeDisturbed
0,1,211480100,cluster6,ICHEC-EC-EARTH,historical,high-structure,repl0,110.483072,264.382737,0.89488,...,-8261.155675,5189.624334,131141.660976,98830.877191,10870.481905,341.416221,86.822206,77.672257,56.831946,0.000000
1,1,211480101,cluster6,ICHEC-EC-EARTH,historical,high-structure,repl0,110.921175,265.921970,0.90784,...,-8184.342041,5326.741280,130881.759593,97911.918345,10280.590935,341.185746,87.013790,77.797130,59.590003,0.000000
2,1,211480102,cluster6,ICHEC-EC-EARTH,historical,high-structure,repl0,110.766483,265.477900,0.90124,...,-8374.620491,4845.040319,130257.791979,98613.682765,10780.541339,341.606554,86.887272,77.733483,54.484221,0.000000
3,1,211480200,cluster6,ICHEC-EC-EARTH,historical,high-structure,repl0,133.923218,327.057380,0.75164,...,-7212.148259,2827.094197,108856.097867,86913.761529,1116.848475,328.489485,123.175890,114.241524,81.385195,0.560437
4,1,211480201,cluster6,ICHEC-EC-EARTH,historical,high-structure,repl0,132.492345,324.006726,0.75796,...,-7326.381045,2798.096820,107133.165195,88219.577467,-84.537791,322.794074,120.495953,112.204182,74.994023,0.000000


In [182]:
len(iland_cluster6['rid'].unique())

12100

# Optimizing one grid cell

In [125]:
iland_one_gc = iland_cluster6[iland_cluster6['rid'] == 211480100]
iland_one_gc = iland_one_gc[iland_one_gc['decade'] == 8][['rid', 'RCPScenario', 'Management', 'abovegroundCarbon', 'shannonIndex', 'evapotranspiration_mm', 'mean_soilwatercontent_mm', 'volumeHarvested']]
iland_one_gc = iland_one_gc.set_index(['rid', 'RCPScenario', 'Management'])

In [126]:
iland_one_gc

abovegroundCarbon  shannonIndex  \
rid       RCPScenario Management                                          
211480100 rcp26       high-structure           168.526375      2.562624   
                      low-structure            153.813628      2.425993   
                      medium-structure         101.190227      3.028314   
                      no-mgmt                  192.715307      3.147355   
          rcp45       high-structure           171.745890      2.793440   
                      low-structure            149.308910      2.509250   
                      medium-structure         102.008893      2.928561   
                      no-mgmt                  178.310006      3.492807   
          rcp85       high-structure           159.092660      2.795987   
                      low-structure            158.287723      2.303972   
                      medium-structure         108.089846      2.910501   
                      no-mgmt                  201.189280      3.198959   

                                        evapotranspiration_mm  \
rid       RCPScenario Management                                
211480100 rcp26       high-structure               368.596023   
                      low-structure                375.128447   
                      medium-structure             361.193677   
                      no-mgmt                      347.883068   
          rcp45       high-structure               351.444644   
                      low-structure                364.362356   
                      medium-structure             304.274209   
                      no-mgmt                      317.159049   
          rcp85       high-structure               329.754771   
                      low-structure                340.872025   
                      medium-structure             311.616346   
                      no-mgmt                      312.513745   

                                        mean_soilwatercontent_mm  \
rid       RCPScenario Management                                   
211480100 rcp26       high-structure                   92.232008   
                      low-structure                    92.239335   
                      medium-structure                 90.672551   
                      no-mgmt                          93.271075   
          rcp45       high-structure                  100.878730   
                      low-structure                   100.435962   
                      medium-structure                102.256095   
                      no-mgmt                          99.993515   
          rcp85       high-structure                   85.760146   
                      low-structure                    87.230546   
                      medium-structure                 84.608741   
                      no-mgmt                          88.029237   

                                        volumeHarvested  
rid       RCPScenario Management                         
211480100 rcp26       high-structure          71.069865  
                      low-structure           30.508732  
                      medium-structure        87.196313  
                      no-mgmt                  0.000000  
          rcp45       high-structure          83.774753  
                      low-structure           14.618366  
                      medium-structure        94.277458  
                      no-mgmt                  0.000000  
          rcp85       high-structure          74.810662  
                      low-structure           34.951186  
                      medium-structure        96.229363  
                      no-mgmt                  0.000000

In [127]:
# TODO: indifference thresholds, checking for 0 division, no "lower is better", add assertions
def normalize(gc_data):
    return (gc_data - gc_data.min()) / (gc_data.max() - gc_data.min())
iland_one_gc_normalized = iland_one_gc.groupby('RCPScenario', group_keys=False).apply(normalize)

# sensitivity analysis with this normalization, not per RCP.
# iland_one_gc_normalized = (iland_one_gc - iland_one_gc.min()) / (iland_one_gc.max() - iland_one_gc.min())

In [128]:
iland_one_gc_normalized

abovegroundCarbon  shannonIndex  \
rid       RCPScenario Management                                          
211480100 rcp26       high-structure             0.735713      0.189406   
                      low-structure              0.574962      0.000000   
                      medium-structure           0.000000      0.834978   
                      no-mgmt                    1.000000      1.000000   
          rcp45       high-structure             0.913971      0.288941   
                      low-structure              0.619913      0.000000   
                      medium-structure           0.000000      0.426322   
                      no-mgmt                    1.000000      1.000000   
          rcp85       high-structure             0.547832      0.549745   
                      low-structure              0.539186      0.000000   
                      medium-structure           0.000000      0.677696   
                      no-mgmt                    1.000000      1.000000   

                                        evapotranspiration_mm  \
rid       RCPScenario Management                                
211480100 rcp26       high-structure                 0.760237   
                      low-structure                  1.000000   
                      medium-structure               0.488546   
                      no-mgmt                        0.000000   
          rcp45       high-structure                 0.785021   
                      low-structure                  1.000000   
                      medium-structure               0.000000   
                      no-mgmt                        0.214432   
          rcp85       high-structure                 0.619997   
                      low-structure                  1.000000   
                      medium-structure               0.000000   
                      no-mgmt                        0.030674   

                                        mean_soilwatercontent_mm  \
rid       RCPScenario Management                                   
211480100 rcp26       high-structure                    0.600132   
                      low-structure                     0.602952   
                      medium-structure                  0.000000   
                      no-mgmt                           1.000000   
          rcp45       high-structure                    0.391241   
                      low-structure                     0.195550   
                      medium-structure                  1.000000   
                      no-mgmt                           0.000000   
          rcp85       high-structure                    0.336620   
                      low-structure                     0.766499   
                      medium-structure                  0.000000   
                      no-mgmt                           1.000000   

                                        volumeHarvested  
rid       RCPScenario Management                         
211480100 rcp26       high-structure           0.815056  
                      low-structure            0.349886  
                      medium-structure         1.000000  
                      no-mgmt                  0.000000  
          rcp45       high-structure           0.888598  
                      low-structure            0.155057  
                      medium-structure         1.000000  
                      no-mgmt                  0.000000  
          rcp85       high-structure           0.777420  
                      low-structure            0.363207  
                      medium-structure         1.000000  
                      no-mgmt                  0.000000

In [132]:
df_long = iland_one_gc_normalized.stack().rename("value")
df_long.index.set_names(['rid', "rcp", "management", "ES"], inplace=True)
df_long = df_long.reorder_levels(["ES", 'rid', "rcp", "management"])
df_final = df_long.unstack("management")

In [133]:
# shape: (#es x #rcps) x #strategies
# TODO there needs to be some testing and assertions for this.
opt.solve_optimization_for_gridcell_general_min_max_distance(df_final.values, ['rcp26', 'rcp45', 'rcp85'])

 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 0.03848521158413572
       x: [ 3.849e-02  5.580e-01  7.123e-02  1.905e-01  1.803e-01]
     nit: 10

# Optimize a dataframe with multiple GCs

In [183]:
iland_multiple_gc = iland_cluster6.copy()
iland_multiple_gc = iland_multiple_gc[iland_multiple_gc['decade'] == 8][['rid', 'RCPScenario', 'Management', 'abovegroundCarbon', 'shannonIndex', 'evapotranspiration_mm', 'mean_soilwatercontent_mm', 'volumeHarvested']]
iland_multiple_gc = iland_multiple_gc.set_index(['rid', 'RCPScenario', 'Management'])
iland_multiple_gc

,,,abovegroundCarbon,shannonIndex,evapotranspiration_mm,mean_soilwatercontent_mm,volumeHarvested
rid,RCPScenario,Management,,,,,
211480100,rcp26,high-structure,168.526375,2.562624,368.596023,92.232008,71.069865
211480101,rcp26,high-structure,168.920009,2.561936,369.306673,91.598207,39.612039
211480102,rcp26,high-structure,180.299373,2.431440,369.148664,92.427082,58.460919
211480200,rcp26,high-structure,179.370778,1.047379,374.376583,91.775040,39.646638
211480201,rcp26,high-structure,201.465909,1.032469,370.219881,92.295714,22.876218
...,...,...,...,...,...,...,...
513280300,rcp85,no-mgmt,200.377371,2.222375,397.866968,119.750200,0.000000
513280301,rcp85,no-mgmt,226.174722,1.368752,384.554335,125.037483,0.000000
513280400,rcp85,no-mgmt,151.492763,4.214620,419.894767,109.035642,0.000000


In [184]:
def convert_to_optimizer_input(gc_data):
    df_long = gc_data.stack().rename("value")
    df_long.index.set_names(['rid', "rcp", "management", "ES"], inplace=True)
    df_long = df_long.reorder_levels(["ES", 'rid', "rcp", "management"])
    return df_long.unstack("management")

TOLERANCE = 0.00001

def prepare_and_optimize_gridcell(gc_data):
    normalized_gc_data = gc_data.groupby('RCPScenario', group_keys=False).apply(normalize)

    optimizer_data = convert_to_optimizer_input(normalized_gc_data)

    opt_result = opt.solve_optimization_for_gridcell_general_min_max_distance(optimizer_data.values, ['rcp26', 'rcp45', 'rcp85'])

    assert opt_result.success == True, "Optimization failed for gc " + str(gc_data.index.get_level_values('rid')[0])
    portfolio_sum = np.sum(opt_result.x[1:])
    assert np.abs(portfolio_sum - 1) < TOLERANCE, "Optimization result does not sum to 1 but " + str(portfolio_sum) + " for gc " + str(gc_data.index.get_level_values('rid')[0])
    return opt_result

In [185]:
prepare_and_optimize_gridcell(iland_one_gc)

 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 0.03848521158413572
       x: [ 3.849e-02  5.580e-01  7.123e-02  1.905e-01  1.803e-01]
     nit: 10

In [186]:
# 1min for 12000 rids, using 1 climate model and 1 replicate
iland_multiple_gc.groupby('rid', group_keys=False).apply(prepare_and_optimize_gridcell)

rid
211480100                            {'x': [0.03848521158413572, 0.5579975315223703, 0.0712349823917093, 0.19048005878745547, 0.18028742729846506], 'fun': 0.03848521158413572, 'slack': [0.013936583406898686, 0.02078123770448153, 0.006777544930398168, 0.001486694747101655, -3.469446951953614e-18, 0.012893962698069483, 0.011052127178668696, 0.008347478654527772, -3.469446951953614e-18, 0.009025934996620077, -5.204170427930421e-18, -5.204170427930421e-18, 0.016498765082886863, 0.0183092767382319, 0.015161993173602794], 'con': [0.0], 'status': 0, 'message': 'Optimization terminated successfully.', 'nit': 10, 'success': True}
211480101                               {'x': [0.0389847134123985, 0.268834604133706, 0.23134838588113607, 0.24407140564184757, 0.25574560434331023], 'fun': 0.0389847134123985, 'slack': [0.011934907796759563, 0.016438189069424437, 0.010532841245040135, 0.008086097073999009, 6.938893903907228e-18, 0.010721891851384165, 0.0143725719241867, 0.0042460214562134654, 6.93

# TODOs for Johannes

- base values --> even for decade == 1 the values differ.
  - obwohl, vllt fuer historical scenario?
- geographical data
- other ESs